In [2]:
!apt-get install -y xvfb python-opengl ffmpeg > /dev/null 2>&1
!pip install pygame pyvirtualdisplay > /dev/null 2>&1

In [3]:
import os
# Setup virtual display environment variables before importing pygame
os.environ["SDL_VIDEODRIVER"] = "dummy"

import pygame
import random
import numpy as np
import IPython.display as ipythondisplay
from pyvirtualdisplay import Display
from pygame.locals import *

# Start the virtual display proxy
v_display = Display(visible=0, size=(800, 600))
v_display.start()

# ==========================================
# GAME INITIALIZATION
# ==========================================
pygame.init()

DIS_WIDTH = 800
DIS_HEIGHT = 600
screen = pygame.display.set_mode((DIS_WIDTH, DIS_HEIGHT))
pygame.display.set_caption("Snake Game - Google Colab Edition")

BLOCK_SIZE = 20
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
RED = (255, 0, 0)
YELLOW = (255, 255, 0)
GREEN = (0, 255, 0)
GREY = (100, 100, 100)

score_font = pygame.font.SysFont("comicsansms", 30)

# Classes matching Lab Activity structural definitions
class SnakeSegment(pygame.sprite.Sprite):
    def __init__(self, x, y):
        super(SnakeSegment, self).__init__()
        self.surf = pygame.Surface((BLOCK_SIZE - 2, BLOCK_SIZE - 2))
        self.surf.fill(GREEN)
        self.rect = self.surf.get_rect()
        self.rect.x = x
        self.rect.y = y

class Fruit(pygame.sprite.Sprite):
    def __init__(self, color):
        super(Fruit, self).__init__()
        self.surf = pygame.Surface((BLOCK_SIZE, BLOCK_SIZE))
        self.surf.fill(color)
        self.rect = self.surf.get_rect()
        self.respawn()

    def respawn(self):
        self.rect.x = round(random.randrange(40, DIS_WIDTH - 40 - BLOCK_SIZE) / 20.0) * 20.0
        self.rect.y = round(random.randrange(40, DIS_HEIGHT - 40 - BLOCK_SIZE) / 20.0) * 20.0

class WallObstacle(pygame.sprite.Sprite):
    def __init__(self, x, y):
        super(WallObstacle, self).__init__()
        self.surf = pygame.Surface((BLOCK_SIZE, BLOCK_SIZE))
        self.surf.fill(GREY)
        self.rect = self.surf.get_rect()
        self.rect.x = x
        self.rect.y = y

# Build Obstacle Map
obstacle_group = pygame.sprite.Group()
wall_coordinates = [
    (200, 200), (220, 200), (240, 200), (260, 200), (280, 200),
    (500, 400), (520, 400), (540, 400), (560, 400), (580, 400)
]
for coord in wall_coordinates:
    obstacle_group.add(WallObstacle(coord[0], coord[1]))

# ==========================================
# SIMULATION ENGINE (Simulates Keyboard Intelligently)
# ==========================================
def run_colab_simulation(max_frames=300):
    x1, y1 = DIS_WIDTH // 2, DIS_HEIGHT // 2
    x1_change, y1_change = BLOCK_SIZE, 0 # Start moving right

    snake_coords_list = [[x1, y1]]
    snake_length = 1
    score = 0

    regular_food = Fruit(RED)
    yellow_food = Fruit(YELLOW)
    yellow_spawned = False

    frames_captured = []

    for frame in range(max_frames):
        # AI/Heuristic Helper: Guide snake towards food automatically since user cannot type live
        target_x = yellow_food.rect.x if yellow_spawned else regular_food.rect.x
        target_y = yellow_food.rect.y if yellow_spawned else regular_food.rect.y

        if x1 < target_x and x1_change == 0:
            x1_change, y1_change = BLOCK_SIZE, 0
        elif x1 > target_x and x1_change == 0:
            x1_change, y1_change = -BLOCK_SIZE, 0
        elif y1 < target_y and y1_change == 0:
            x1_change, y1_change = 0, BLOCK_SIZE
        elif y1 > target_y and y1_change == 0:
            x1_change, y1_change = 0, -BLOCK_SIZE

        x1 += x1_change
        y1 += y1_change

        # Boundary Reset safety rule for simulation continuity
        if x1 >= DIS_WIDTH or x1 < 0 or y1 >= DIS_HEIGHT or y1 < 0:
            x1, y1 = DIS_WIDTH // 2, DIS_HEIGHT // 2

        snake_coords_list.append([x1, y1])
        if len(snake_coords_list) > snake_length:
            del snake_coords_list[0]

        # Process standard food collision
        if x1 == regular_food.rect.x and y1 == regular_food.rect.y:
            regular_food.respawn()
            snake_length += 1
            score += 15  # Reaches 100+ and 150+ thresholds quickly

        # Process features conditionally (Task 2 Speed & Items)
        if score > 150:
            if not yellow_spawned:
                yellow_food.respawn()
                yellow_spawned = True
            if x1 == yellow_food.rect.x and y1 == yellow_food.rect.y:
                yellow_food.respawn()
                snake_length += 2
                score += 30
        else:
            yellow_spawned = False

        # Rendering
        screen.fill(BLACK)
        for block in obstacle_group:
            screen.blit(block.surf, block.rect)
        screen.blit(regular_food.surf, regular_food.rect)
        if yellow_spawned:
            screen.blit(yellow_food.surf, yellow_food.rect)
        for coord in snake_coords_list:
            segment = SnakeSegment(coord[0], coord[1])
            screen.blit(segment.surf, segment.rect)

        # Draw HUD info
        score_surface = score_font.render(f"Score: {score} | Frame: {frame}", True, WHITE)
        screen.blit(score_surface, (15, 15))

        pygame.display.flip()

        # Capture frame pixel data for compilation
        view = pygame.surfarray.array3d(screen)
        view = view.transpose([1, 0, 2]) # Align dimensions correctly
        frames_captured.append(view)

    pygame.quit()
    v_display.stop()
    return frames_captured

# Execute simulation loop capture
print("Running simulation and capturing frames...")
video_frames = run_colab_simulation(max_frames=250)
print(f"Captured {len(video_frames)} frames successfully!")

pygame 2.6.1 (SDL 2.28.4, Python 3.12.13)
Hello from the pygame community. https://www.pygame.org/contribute.html
Running simulation and capturing frames...
Captured 250 frames successfully!


In [4]:
import matplotlib.pyplot as plt
from matplotlib import animation

def display_video(frames):
    fig = plt.figure(figsize=(8, 6))
    plt.axis('off')

    # Initialize placeholder plot image block
    im = plt.imshow(frames[0])

    def animate(i):
        im.set_data(frames[i])
        return [im]

    anim = animation.FuncAnimation(fig, animate, frames=len(frames), interval=50, blit=True)
    video = anim.to_html5_video()
    html = ipythondisplay.HTML(video)
    ipythondisplay.display(html)
    plt.close()

# Render inline video player widget
display_video(video_frames)